# FD004 - Modelado final interno

Objetivo: plantear y evaluar un modelo para FD004 usando lo aprendido en FD002 y FD003. FD004 combina seis condiciones operativas con dos patrones de degradacion posibles, por eso el pipeline controla primero la condicion y despues agrega features temporales fault-sensitive.


## Lectura metodologica

- FD002 enseno que mezclar condiciones operativas rompe la senal sensor-RUL. Hay que inferir `condition_id`, agregar one-hot por condicion y normalizar sensores dentro de cada condicion.
- FD003 enseno que, cuando hay patrones de degradacion distintos, la mejora real viene de pendientes, deltas, volatilidad, aceleracion e interacciones entre sensores, no de usar clusters post-hoc como etiqueta.
- FD004 necesita ambas cosas. Los clusters residuales del EDA pueden orientar sensores, pero no se usan como feature directa porque se calculan con trayectoria completa y podrian filtrar futuro.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parents[2] if Path.cwd().name == 'modeling' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / 'results' / 'FD004'
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'FD004' / 'fd004_best_model_config.json'
NOTES_PATH = PROJECT_ROOT / 'notas' / 'hallazgos' / 'FD004' / 'fd004_modeling_interpretation.txt'

comparison = pd.read_csv(RESULTS_DIR / 'fd004_model_family_comparison.csv')
search = pd.read_csv(RESULTS_DIR / 'fd004_hyperparam_search.csv')
finalist_summary = pd.read_csv(RESULTS_DIR / 'fd004_finalist_multisplit_summary.csv')
validation_metrics = pd.read_csv(RESULTS_DIR / 'fd004_best_validation_metrics.csv')
validation_bins = pd.read_csv(RESULTS_DIR / 'fd004_best_validation_metrics_by_rul_bin.csv')
official_metrics = pd.read_csv(RESULTS_DIR / 'fd004_official_test_metrics.csv')
with open(CONFIG_PATH, 'r', encoding='utf-8') as file:
    best_config = json.load(file)


## Transferencia desde FD002 y FD003

La comparacion inicial muestra que FD004 no debe tratarse como FD003 crudo: ignorar condiciones sube mucho el C-MAPSS. Tambien muestra que sumar features fault-sensitive mejora sobre la representacion solo condition-normalized.


In [ ]:
comparison[[
    'candidate_label', 'model_type', 'feature_set', 'window_size',
    'sample_weight_scheme', 'cmapss_score', 'rmse', 'dangerous_error_pct'
]].style.format({
    'cmapss_score': '{:.3f}',
    'rmse': '{:.3f}',
    'dangerous_error_pct': '{:.2f}',
})


## Busqueda acotada de hiperparametros

La busqueda se centro en la representacion `condition_fault_sensitive`. Antes de tunear se fijo la direccion metodologica: controlar conditions como FD002, agregar degradacion reciente como FD003, y comparar XGBoost/LightGBM solo en variantes plausibles.


In [ ]:
search[[
    'candidate_label', 'model_type', 'feature_set', 'window_size',
    'sample_weight_scheme', 'cmapss_score', 'rmse', 'dangerous_error_pct'
]].style.format({
    'cmapss_score': '{:.3f}',
    'rmse': '{:.3f}',
    'dangerous_error_pct': '{:.2f}',
})


## Finalistas multi-split

El mejor candidato del split 42 se probo contra otros finalistas en tres splits. Esto evita cerrar el modelo por una particion particularmente favorable.


In [ ]:
finalist_summary[[
    'candidate_label', 'model_type', 'feature_set', 'window_size', 'sample_weight_scheme',
    'mean_cmapss_score', 'std_cmapss_score', 'worst_cmapss_score',
    'mean_rmse', 'mean_dangerous_error_pct', 'worst_dangerous_error_pct'
]].style.format({
    'mean_cmapss_score': '{:.3f}',
    'std_cmapss_score': '{:.3f}',
    'worst_cmapss_score': '{:.3f}',
    'mean_rmse': '{:.3f}',
    'mean_dangerous_error_pct': '{:.2f}',
    'worst_dangerous_error_pct': '{:.2f}',
})


## Modelo final

El candidato seleccionado es `fd004_xgb_fs_bin_weights_w70`: XGBoost con features condition-fault-sensitive, ventana de 70 ciclos y `bin_weights`. La ventana mas larga fue la mejora concreta frente a transferir directamente el XGB final de FD002.


In [ ]:
best_config['final_model']


In [ ]:
{
    'feature_set': best_config['preprocessing']['feature_set'],
    'window_size': best_config['preprocessing']['window_size'],
    'n_features': best_config['preprocessing']['n_features'],
    'fault_sensitive_sensors': best_config['preprocessing']['fault_sensitive_sensors'],
}


## Metricas finales

La validacion interna selecciona el modelo. El test oficial se reporta solo despues de esa seleccion, igual que en FD002 y FD003.


In [ ]:
display(validation_metrics)
display(official_metrics)


In [ ]:
validation_bins[['rul_bin', 'n_eval', 'mae', 'rmse', 'cmapss_score', 'dangerous_error_pct']].style.format({
    'mae': '{:.3f}',
    'rmse': '{:.3f}',
    'cmapss_score': '{:.3f}',
    'dangerous_error_pct': '{:.2f}',
})


## Regeneracion opcional

La celda siguiente ejecuta el workflow completo. Puede tardar mucho porque entrena comparaciones, busqueda, finalistas, modelo final y test oficial.


In [ ]:
# Regeneracion opcional, no necesaria para leer el notebook.
# from src.fd004_modeling import run_fd004_modeling_workflow
# result = run_fd004_modeling_workflow(
#     data_dir='CMAPSSData',
#     random_state=42,
#     finalist_random_states=(0, 1, 2),
#     finalist_count=4,
# )
# result['finalist_summary'].head()


## Artefactos

- `src/fd004_modeling.py`
- `results/FD004/fd004_model_family_comparison.csv`
- `results/FD004/fd004_hyperparam_search.csv`
- `results/FD004/fd004_finalist_multisplit_summary.csv`
- `configs/FD004/fd004_best_model_config.json`
- `predictions/fd004_best_model_predictions.csv`
- `checkpoints/fd004_best_model.joblib`
